## Setup: load data into SQLite

In [37]:
import pandas as pd
import sqlite3
import os

# Load raw CSV directly from Downloads
df = pd.read_csv('/Users/shraddhaingle/Downloads/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Clean it
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

# Save database in Downloads too
conn = sqlite3.connect('/Users/shraddhaingle/Downloads/telco_churn.db')
df.to_sql('customers', conn, if_exists='replace', index=False)

print(f"✅ Done — {len(df):,} rows loaded")

✅ Done — 7,043 rows loaded


In [38]:
def run_sql(query):
    return pd.read_sql(query, conn)

## Query 1: Overall KPIs

In [39]:
run_sql("""
SELECT
    COUNT(*)                                                AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)         AS churned,
    SUM(CASE WHEN Churn = 'No'  THEN 1 ELSE 0 END)         AS retained,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1)
                                                            AS churn_rate_pct,
    ROUND(AVG(MonthlyCharges), 2)                           AS avg_monthly_charge,
    ROUND(SUM(MonthlyCharges), 0)                           AS total_mrr
FROM customers
""")

,total_customers,churned,retained,churn_rate_pct,avg_monthly_charge,total_mrr
0,7043,1869,5174,26.5,64.76,456117.0


 ## Query 2: Churn by Contract Type

In [40]:
run_sql("""
SELECT
    Contract,
    COUNT(*)                                                AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)         AS churned,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1)
                                                            AS churn_rate_pct,
    ROUND(AVG(MonthlyCharges), 2)                           AS avg_monthly_charge
FROM customers
GROUP BY Contract
ORDER BY churn_rate_pct DESC
""")

,Contract,total_customers,churned,churn_rate_pct,avg_monthly_charge
0,Month-to-month,3875,1655,42.7,66.40
1,One year,1473,166,11.3,65.05
2,Two year,1695,48,2.8,60.77


##  Query 3: Churn by Tenure Bucket

In [41]:
run_sql("""
SELECT
    CASE
        WHEN tenure BETWEEN  1 AND  3 THEN '1. New       (1-3 mo)'
        WHEN tenure BETWEEN  4 AND 12 THEN '2. Early     (4-12 mo)'
        WHEN tenure BETWEEN 13 AND 24 THEN '3. Growing  (13-24 mo)'
        WHEN tenure BETWEEN 25 AND 48 THEN '4. Mature   (25-48 mo)'
        ELSE                               '5. Loyal     (49+ mo)'
    END                                                     AS tenure_bucket,
    COUNT(*)                                                AS customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)         AS churned,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1)
                                                            AS churn_rate_pct,
    ROUND(AVG(MonthlyCharges), 2)                           AS avg_charges
FROM customers
GROUP BY tenure_bucket
ORDER BY tenure_bucket
""")

,tenure_bucket,customers,churned,churn_rate_pct,avg_charges
0,1. New (1-3 mo),1051,597,56.8,53.44
1,2. Early (4-12 mo),1124,440,39.1,58.73
2,3. Growing (13-24 mo),1024,294,28.7,61.36
3,4. Mature (25-48 mo),1594,325,20.4,65.93
4,5. Loyal (49+ mo),2250,213,9.5,73.79


## Query 4: Revenue at Risk

In [42]:
run_sql("""
SELECT
    Contract,
    InternetService,
    COUNT(*)                                                AS at_risk_customers,
    ROUND(SUM(MonthlyCharges), 0)                           AS mrr_at_risk,
    ROUND(AVG(MonthlyCharges), 2)                           AS avg_charge,
    ROUND(AVG(tenure), 1)                                   AS avg_tenure_months
FROM customers
WHERE
    Churn              = 'No'
    AND Contract       = 'Month-to-month'
    AND tenure         < 12
    AND OnlineSecurity = 'No'
    AND TechSupport    = 'No'
GROUP BY Contract, InternetService
ORDER BY mrr_at_risk DESC
""")

,Contract,InternetService,at_risk_customers,mrr_at_risk,avg_charge,avg_tenure_months
0,Month-to-month,Fiber optic,195,15308.0,78.50,5.1
1,Month-to-month,DSL,203,9341.0,46.01,4.3


* 24,649(dollar) in monthly  revenue  is  at  immediate  risk — 398 active customers who are new (under 12 months), on month-to-month contracts, with no security or support add-ons. These customers look identical to the ones who already churned. A targeted retention campaign on just this group protects $295K in annual revenue.
 
* Fiber optic customers are more expensive to lose — 195 fiber customers at 78.50(dollar)/month average vs 203 DSL customers at $46/month. Fiber churners cost nearly double per lost customer, making them the priority segment for the customer success team to call first.



##  Query 5: Churn by Internet + Add-ons

In [43]:
run_sql("""
SELECT
    InternetService,
    OnlineSecurity,
    TechSupport,
    COUNT(*)                                                AS customers,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1)
                                                            AS churn_rate_pct
FROM customers
WHERE InternetService != 'No'
GROUP BY InternetService, OnlineSecurity, TechSupport
HAVING COUNT(*) > 50
ORDER BY churn_rate_pct DESC
LIMIT 10
""")

,InternetService,OnlineSecurity,TechSupport,customers,churn_rate_pct
0,Fiber optic,No,No,1765,55.0
1,DSL,No,No,788,35.4
2,Fiber optic,No,Yes,492,29.1
3,Fiber optic,Yes,No,465,28.0
4,DSL,No,Yes,453,15.0
5,DSL,Yes,No,455,14.5
6,Fiber optic,Yes,Yes,374,14.2
7,DSL,Yes,Yes,725,6.3


## Query 6: Payment Method Risk

In [44]:
run_sql("""
SELECT
    PaymentMethod,
    COUNT(*)                                                AS customers,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1)
                                                            AS churn_rate_pct,
    ROUND(AVG(MonthlyCharges), 2)                           AS avg_charges
FROM customers
GROUP BY PaymentMethod
ORDER BY churn_rate_pct DESC
""")

,PaymentMethod,customers,churn_rate_pct,avg_charges
0,Electronic check,2365,45.3,76.26
1,Mailed check,1612,19.1,43.92
2,Bank transfer (automatic),1544,16.7,67.19
3,Credit card (automatic),1522,15.2,66.51


In [45]:
##  Close the connection
conn.close()
